# Baseline Models for Rent Price Prediction

This notebook demonstrates the creation of baseline models for rent price prediction.

**Key steps:**
- Data loading and feature selection
- Log-transforming the target variable (`price`) for better model performance
- Using cross-validation to evaluate models
- Comparing baseline models using RMSLE metric (on original price scale), because RMSLE penalized the underestimate more than overestimate

## Imports

In [51]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.metrics import root_mean_squared_log_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

from src.models.model_evaluation import run_cv

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

## Load Data and Select Features

In [54]:
data = pd.read_csv('../data/processed/train_df.csv')

In [25]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9140 entries, 0 to 9139
Data columns (total 28 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              9140 non-null   int64  
 1   price           9140 non-null   int64  
 2   address         9140 non-null   object 
 3   coordinates     9032 non-null   object 
 4   region          9140 non-null   object 
 5   subway          7375 non-null   object 
 6   rooms           9140 non-null   int64  
 7   footage         9140 non-null   object 
 8   floor           9140 non-null   int64  
 9   features        9140 non-null   object 
 10  residential     4367 non-null   object 
 11  neighborhood    9140 non-null   object 
 12  description     9140 non-null   object 
 13  detail          9140 non-null   object 
 14  attributes      5501 non-null   object 
 15  full_area       9140 non-null   float64
 16  living_area     9140 non-null   float64
 17  kitchen_area    9140 non-null   f

In [26]:
black_list = ['id', 'price', 'price_bin']

numerical_feats = [col for col in data.select_dtypes(exclude='object').columns
                   if col not in black_list]

In [27]:
data[numerical_feats].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9140 entries, 0 to 9139
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   rooms         9140 non-null   int64  
 1   floor         9140 non-null   int64  
 2   full_area     9140 non-null   float64
 3   living_area   9140 non-null   float64
 4   kitchen_area  9140 non-null   float64
 5   num_storeys   9140 non-null   int64  
 6   lon           9032 non-null   float64
 7   lat           9032 non-null   float64
dtypes: float64(5), int64(3)
memory usage: 571.4 KB


In [28]:
X = data[numerical_feats]
y = data['price']

## Baseline Models

In [31]:
results = {}

### DummyRegressor (mean)

In [53]:
model = DummyRegressor(strategy='mean')
mean_score = run_cv(model, X, y)
results['DummyRegressor-mean'] = mean_score

[Fold 0] train_rmsle: 0.7376, val_rmsle: 0.7320
[Fold 1] train_rmsle: 0.7351, val_rmsle: 0.7373
[Fold 2] train_rmsle: 0.7344, val_rmsle: 0.7385
RMSLE: 0.7359 ± 0.0028


### DummyRegressor (median)

In [33]:
model = DummyRegressor(strategy='median')
median_score = run_cv(model, X, y)
results['DummyRegressor-median'] = median_score

[Fold 0] train_rmsle: 0.7414, val_rmsle: 0.7368
[Fold 1] train_rmsle: 0.7404, val_rmsle: 0.7387
[Fold 2] train_rmsle: 0.7378, val_rmsle: 0.7440
RMSLE: 0.7398 ± 0.0030


### Linear Regression

In [34]:
pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

lr_score = run_cv(pipeline, X, y)
results['LinearRegression'] = lr_score

[Fold 0] train_rmsle: 0.4874, val_rmsle: 0.4779
[Fold 1] train_rmsle: 0.4785, val_rmsle: 0.4961
[Fold 2] train_rmsle: 0.4857, val_rmsle: 0.4814
RMSLE: 0.4851 ± 0.0079


### ExtraTreesRegressor

In [35]:
pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('model', ExtraTreesRegressor(n_jobs=-1, random_state=7))
])

et_score = run_cv(pipeline, X, y)
results['ExtraTreesRegressor'] = et_score

[Fold 0] train_rmsle: 0.0060, val_rmsle: 0.2945
[Fold 1] train_rmsle: 0.0062, val_rmsle: 0.3046
[Fold 2] train_rmsle: 0.0071, val_rmsle: 0.3039
RMSLE: 0.3010 ± 0.0046


### RandomForestRegressor

In [36]:
pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('model', RandomForestRegressor(n_jobs=-1, random_state=7))
])

rf_score = run_cv(pipeline, X, y)
results['RandomForestRegressor'] = rf_score

[Fold 0] train_rmsle: 0.1106, val_rmsle: 0.2853
[Fold 1] train_rmsle: 0.1089, val_rmsle: 0.2978
[Fold 2] train_rmsle: 0.1085, val_rmsle: 0.2968
RMSLE: 0.2933 ± 0.0057


### XGBoost

In [ ]:
parameters_xgb = {
    'verbosity': 1,
    'seed': 7,
}
model = xgb.XGBRegressor(**parameters_xgb)

xgb_score = run_cv(model, X, y)
results['XGBRegressor'] = xgb_score

[Fold 0] train_rmsle: 0.1438, val_rmsle: 0.2779
[Fold 1] train_rmsle: 0.1437, val_rmsle: 0.2870
[Fold 2] train_rmsle: 0.1363, val_rmsle: 0.2935
RMSLE: 0.2862 ± 0.0064


### LightGBM

In [ ]:
parameters_lgb = {
    'random_state': 7,
    'verbose': 0,
}

model = lgb.LGBMRegressor(**parameters_lgb)

lgb_score = run_cv(model, X, y)
results['LGBMRegressor'] = lgb_score

[Fold 0] train_rmsle: 0.2311, val_rmsle: 0.2852
[Fold 1] train_rmsle: 0.2276, val_rmsle: 0.2917
[Fold 2] train_rmsle: 0.2237, val_rmsle: 0.2943
RMSLE: 0.2904 ± 0.0038


## Summary Table

In [50]:
df_results = pd.DataFrame(results.items(), columns=['Model', 'RMSLE'])
df_results = df_results.sort_values('RMSLE')
df_results.style.highlight_min(subset=['RMSLE'], color='tan')

,Model,RMSLE
5,XGBRegressor,0.286153
6,LGBMRegressor,0.290385
4,RandomForestRegressor,0.293310
3,ExtraTreesRegressor,0.301003
2,LinearRegression,0.485117
0,DummyRegressor-mean,0.735939
1,DummyRegressor-median,0.739846


- We established several baseline models for rent price prediction using only numerical features and log-transformed target values.
- Tree-based models (RandomForest, ExtraTrees, XGBoost, LightGBM) significantly outperform simple baselines (Dummy, Linear Regression) in terms of RMSLE.
- The best baseline RMSLE is achieved by XGBoost (0.286), followed closely by LightGBM (0.290) and RandomForest (0.293).

**Conclusions:**
- Keep DummyRegressors as baseline references for future experiments.
- Perform feature engineering, incorporate categorical, text, and geospatial features.
- Proceed with hyperparameter tuning for XGB, LGBM and Random Forest as the leading baselines to further improve performance.